# Human Images Background Removal (ResNet-34 U-Net)
### Clean, Step-by-Step Training & Inference on Kaggle

This simplified notebook trains a human portrait background removal model:
- **Codebase**: Flat, easy-to-read structure (`config.py`, `dataset.py`, `model.py`, `loss.py`, `train.py`)
- **Dataset**: P3M-10k or portrait matting pairs
- **Hardware**: Kaggle GPU (T4 or P100), Internet ON

## 1. Clone Repository & Setup Working Directory

In [ ]:
import os, sys
REPO_NAME = "bg-removal-with-deep-learning"
REPO_URL = "https://github.com/Razamindset/bg-removal-with-deep-learning.git"

if not os.path.isdir("model.py") and not os.path.isfile("model.py"):
    if not os.path.isdir(REPO_NAME):
        !git clone {REPO_URL}
    if os.path.isdir(REPO_NAME):
        os.chdir(REPO_NAME)

if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

print("Working Directory:", os.getcwd())
print("Files:", [f for f in os.listdir(".") if not f.startswith(".")])

## 2. Install Requirements & Pre-cache ResNet-34 Weights

In [ ]:
!pip install -q -r requirements.txt

import torchvision.models as models
print("Pre-loading ResNet-34 weights into cache...")
_ = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
print("Weights cached successfully!")

## 3. Verify Config & Dataset
Edit `config.py` or override paths directly below:

In [ ]:
from config import cfg
# If dataset is in /kaggle/input/..., adjust data_dir if needed:
# cfg.data_dir = "/kaggle/input/p3m-10k"
cfg.display()

from dataset import get_dataloaders
train_loader, val_loader = get_dataloaders(cfg)
images, masks = next(iter(train_loader))
print(f"Sample Batch - Images: {images.shape}, Masks: {masks.shape}")

## 4. Run Training
Trains ResNetUNet with CosineAnnealingLR, backbone freezing warmup, and early stopping.

In [ ]:
!python train.py

## 5. Plot Training Loss & Accuracy Curves

In [ ]:
!python plot.py --csv outputs/history.csv --out outputs/curves.png

## 6. Run Inference & Display Visual Cutouts

In [ ]:
import glob, os, matplotlib.pyplot as plt
from PIL import Image

# Run prediction on first few images
sample_imgs = glob.glob(f"{cfg.data_dir}/**/*.jpg", recursive=True)[:3]
if not sample_imgs:
    sample_imgs = glob.glob(f"{cfg.data_dir}/**/*.png", recursive=True)[:3]

for img_p in sample_imgs:
    !python predict.py --weights outputs/best_model.pth --input {img_p} --output outputs/predictions

# Display Results
pred_files = glob.glob("outputs/predictions/*_nobg.png")[:3]
fig, axes = plt.subplots(len(pred_files), 2, figsize=(10, 4 * len(pred_files)))
for i, p_file in enumerate(pred_files):
    base = os.path.basename(p_file).replace("_nobg.png", "")
    orig_p = [s for s in sample_imgs if base in s][0]
    axes[i, 0].imshow(Image.open(orig_p))
    axes[i, 0].set_title("Original Image")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(Image.open(p_file))
    axes[i, 1].set_title("Background Removed (RGBA)")
    axes[i, 1].axis("off")
plt.tight_layout()
plt.show()

## 7. Package Artifacts for 1-Click Download

In [ ]:
from IPython.display import FileLink
!zip -rq /kaggle/working/human_bg_removal_artifacts.zip outputs
display(FileLink("human_bg_removal_artifacts.zip"))